In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [34]:
!pip install -q pandas numpy scikit-learn streamlit pyngrok
print("Libraries installed successfully!")

Libraries installed successfully!


In [35]:
# Load dataset
df = pd.read_csv('/content/Thales_Group_Manufacturing.csv')

In [36]:
print("Initial Dataset Shape:", df.shape)

Initial Dataset Shape: (100000, 14)


In [37]:
# Check basic info and missing values
print(df.info())
print("Missing values per column:\n", df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Date                           100000 non-null  object 
 1   Timestamp                      100000 non-null  object 
 2   Machine_ID                     100000 non-null  int64  
 3   Operation_Mode                 100000 non-null  object 
 4   Temperature_C                  100000 non-null  float64
 5   Vibration_Hz                   100000 non-null  float64
 6   Power_Consumption_kW           100000 non-null  float64
 7   Network_Latency_ms             100000 non-null  float64
 8   Packet_Loss_%                  100000 non-null  float64
 9   Quality_Control_Defect_Rate_%  100000 non-null  float64
 10  Production_Speed_units_per_hr  100000 non-null  float64
 11  Predictive_Maintenance_Score   100000 non-null  float64
 12  Error_Rate_%                   

In [38]:
# Drop complete duplicate rows if any exist
df.drop_duplicates(inplace=True)

In [39]:
# Handle missing values (imputation)
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include=['object']).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

In [40]:
# Combine Date and Timestamp into a datetime object and sort
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Timestamp'], format='%d-%m-%Y %H:%M:%S')
df = df.sort_values('Datetime').reset_index(drop=True)

print("Data Cleaning & Sorting Completed. Clean shape:", df.shape)

Data Cleaning & Sorting Completed. Clean shape: (100000, 15)


In [41]:
# Feature Engineering based on project guidelines
# 1. Sensor stability indicators
df['Sensor_Stability_Index'] = (df['Temperature_C'] / 100.0) * (df['Vibration_Hz'] + 1)

In [42]:
# 2. Energy efficiency ratios (units produced per kW)
df['Energy_Efficiency_Ratio'] = df['Production_Speed_units_per_hr'] / (df['Power_Consumption_kW'] + 1e-5)

In [43]:
# 3. Error-to-output ratios
df['Error_Output_Ratio'] = df['Error_Rate_%'] / (df['Production_Speed_units_per_hr'] + 1e-5)

In [44]:
# 4. Network reliability score (combining latency and packet loss)
df['Network_Reliability_Score'] = 100 - (df['Network_Latency_ms'] * 0.1 + df['Packet_Loss_%'] * 2)

print("Feature Engineering completed successfully.")
print(df[['Sensor_Stability_Index', 'Energy_Efficiency_Ratio', 'Network_Reliability_Score']].head())

Feature Engineering completed successfully.
   Sensor_Stability_Index  Energy_Efficiency_Ratio  Network_Reliability_Score
0                3.336951                55.464055                    98.5189
1                3.670583               175.484022                    92.6328
2                1.363824                17.590303                    94.8863
3                0.526586                81.017254                    94.7626
4                1.010361                25.556335                    87.0031


In [45]:
# Encoding categorical columns
le_op = LabelEncoder()
df['Operation_Mode_Encoded'] = le_op.fit_transform(df['Operation_Mode'])

le_target = LabelEncoder()
df['Efficiency_Status_Encoded'] = le_target.fit_transform(df['Efficiency_Status'])

In [46]:
# Select features
feature_cols = [
    'Machine_ID', 'Operation_Mode_Encoded', 'Temperature_C', 'Vibration_Hz',
    'Power_Consumption_kW', 'Network_Latency_ms', 'Packet_Loss_%',
    'Quality_Control_Defect_Rate_%', 'Production_Speed_units_per_hr',
    'Predictive_Maintenance_Score', 'Error_Rate_%',
    'Sensor_Stability_Index', 'Energy_Efficiency_Ratio',
    'Error_Output_Ratio', 'Network_Reliability_Score'
]

X = df[feature_cols]
y = df['Efficiency_Status_Encoded']

In [47]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [48]:
# Train Random Forest Classifier
print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

Training Random Forest Classifier...


RandomForestClassifier(random_state=42)

In [49]:
# Evaluate model
y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le_target.classes_))

Accuracy: 0.9998

Classification Report:
               precision    recall  f1-score   support

        High       1.00      1.00      1.00       597
         Low       1.00      1.00      1.00     15565
      Medium       1.00      1.00      1.00      3838

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



In [50]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

st.set_page_config(page_title="Manufacturing Efficiency Dashboard", page_icon="🏭", layout="wide")

@st.cache_resource
def load_data_and_model():
    df = pd.read_csv('Thales_Group_Manufacturing.csv')
    df.drop_duplicates(inplace=True)

    df['Sensor_Stability_Index'] = (df['Temperature_C'] / 100.0) * (df['Vibration_Hz'] + 1)
    df['Energy_Efficiency_Ratio'] = df['Production_Speed_units_per_hr'] / (df['Power_Consumption_kW'] + 1e-5)
    df['Error_Output_Ratio'] = df['Error_Rate_%'] / (df['Production_Speed_units_per_hr'] + 1e-5)
    df['Network_Reliability_Score'] = 100 - (df['Network_Latency_ms'] * 0.1 + df['Packet_Loss_%'] * 2)

    le_op = LabelEncoder()
    df['Operation_Mode_Encoded'] = le_op.fit_transform(df['Operation_Mode'])

    le_target = LabelEncoder()
    df['Efficiency_Status_Encoded'] = le_target.fit_transform(df['Efficiency_Status'])

    feature_cols = [
        'Machine_ID', 'Operation_Mode_Encoded', 'Temperature_C', 'Vibration_Hz',
        'Power_Consumption_kW', 'Network_Latency_ms', 'Packet_Loss_%',
        'Quality_Control_Defect_Rate_%', 'Production_Speed_units_per_hr',
        'Predictive_Maintenance_Score', 'Error_Rate_%',
        'Sensor_Stability_Index', 'Energy_Efficiency_Ratio',
        'Error_Output_Ratio', 'Network_Reliability_Score'
    ]

    X = df[feature_cols]
    y = df['Efficiency_Status_Encoded']

    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X, y)

    return df, model, le_op, le_target, feature_cols

df, model, le_op, le_target, feature_cols = load_data_and_model()

st.title("🏭 AI-Based Manufacturing Efficiency Classification Dashboard")

st.sidebar.header("🎛️ Factory Controls")
selected_machine = st.sidebar.selectbox("Select Machine ID", sorted(df['Machine_ID'].unique()))
selected_op_mode = st.sidebar.selectbox("Operation Mode", le_op.classes_)

temp = st.sidebar.slider("Temperature (°C)", float(df['Temperature_C'].min()), float(df['Temperature_C'].max()), float(df['Temperature_C'].mean()))
vibration = st.sidebar.slider("Vibration (Hz)", float(df['Vibration_Hz'].min()), float(df['Vibration_Hz'].max()), float(df['Vibration_Hz'].mean()))
power = st.sidebar.slider("Power Consumption (kW)", float(df['Power_Consumption_kW'].min()), float(df['Power_Consumption_kW'].max()), float(df['Power_Consumption_kW'].mean()))
latency = st.sidebar.slider("Network Latency (ms)", float(df['Network_Latency_ms'].min()), float(df['Network_Latency_ms'].max()), float(df['Network_Latency_ms'].mean()))
packet_loss = st.sidebar.slider("Packet Loss (%)", float(df['Packet_Loss_%'].min()), float(df['Packet_Loss_%'].max()), float(df['Packet_Loss_%'].mean()))
defect_rate = st.sidebar.slider("Defect Rate (%)", float(df['Quality_Control_Defect_Rate_%'].min()), float(df['Quality_Control_Defect_Rate_%'].max()), float(df['Quality_Control_Defect_Rate_%'].mean()))
prod_speed = st.sidebar.slider("Production Speed (units/hr)", float(df['Production_Speed_units_per_hr'].min()), float(df['Production_Speed_units_per_hr'].max()), float(df['Production_Speed_units_per_hr'].mean()))
maintenance_score = st.sidebar.slider("Predictive Maintenance Score", float(df['Predictive_Maintenance_Score'].min()), float(df['Predictive_Maintenance_Score'].max()), float(df['Predictive_Maintenance_Score'].mean()))
error_rate = st.sidebar.slider("Error Rate (%)", float(df['Error_Rate_%'].min()), float(df['Error_Rate_%'].max()), float(df['Error_Rate_%'].mean()))

op_encoded = le_op.transform([selected_op_mode])[0]
sensor_stability = (temp / 100.0) * (vibration + 1)
energy_eff = prod_speed / (power + 1e-5)
error_output = error_rate / (prod_speed + 1e-5)
network_reliability = 100 - (latency * 0.1 + packet_loss * 2)

input_data = pd.DataFrame([[
    selected_machine, op_encoded, temp, vibration, power, latency,
    packet_loss, defect_rate, prod_speed, maintenance_score, error_rate,
    sensor_stability, energy_eff, error_output, network_reliability
]], columns=feature_cols)

prediction_encoded = model.predict(input_data)[0]
prediction_proba = model.predict_proba(input_data)[0]
predicted_label = le_target.inverse_transform([prediction_encoded])[0]
confidence = prediction_proba[prediction_encoded] * 100

st.markdown("---")
col1, col2 = st.columns(2)

with col1:
    st.subheader("⚡ Efficiency Prediction")
    if predicted_label == 'High':
        st.success(f"**Predicted State:** {predicted_label}")
    elif predicted_label == 'Medium':
        st.warning(f"**Predicted State:** {predicted_label}")
    else:
        st.error(f"**Predicted State:** {predicted_label}")
    st.metric("Prediction Confidence", f"{confidence:.2f}%")

with col2:
    st.subheader("📊 Class Probabilities")
    proba_df = pd.DataFrame({'Class': le_target.classes_, 'Probability': prediction_proba})
    st.bar_chart(proba_df.set_index('Class'))

Overwriting app.py


In [63]:
import subprocess
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])
print("Streamlit app ready !")

Streamlit app ready !


In [ ]:
# Localtunnel install
!npm install -g localtunnel

# Port 8501 link generate
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴⠙⠹⠸⠼⠴your url is: https://late-chefs-decide.loca.lt
